In [ ]:
%pip install scrubadub sklearn_crfsuite

import os
import tensorflow as tf
import numpy as np
import nltk
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from typing import Set
import re
import json
import scrubadub
import joblib
import sklearn_crfsuite

# Download NLTK stopwords if necessary
nltk.download('stopwords')
from nltk.corpus import stopwords

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Import necessary libraries
from pyspark.sql import SparkSession
import pandas as pd

# Set up Spark session
spark = SparkSession.builder \
    .appName("Verbatims") \
    .getOrCreate()

# Define path to your data in Azure Data Lake Storage
data_path = " Verbatim and Topics v3.xlsx"

# Load data into a Spark DataFrame
df = spark.read.format("com.crealytics.spark.excel").option("header", "true").load(data_path)

# Load Spark DF to Pandas DF
pandas_df = df.toPandas()

display(pandas_df)

df = pandas_df.copy()  # create a copy to avoid modifying the original DataFrame

df.columns = ['o_comment', 'topic']

In [ ]:
# Implementing stopwords
stopwords = set(stopwords.words('english'))
custom_stop_words = { 'water'}
stop_words = stopwords.union(custom_stop_words)

In [ ]:
def preprocess_text(text):
    text = text.lower().strip()  # lowercasing and stripping whitespace
    text = scrubadub.clean(text) # removing PII
    
    tokens = text.split()
    filtered_tokens = [token for token in tokens if token not in stop_words] # removing stopwords
    text = " ".join(filtered_tokens)
    return text

# Preprocess comments and add special tokens for start and end of sentence.
# Also, we use a special token (<oov>) for out-of-vocabulary words.
df['comment'] = df['o_comment'].apply(preprocess_text)
df['comment'] = df['comment'].apply(lambda x: "<start> " + x + " <end>")

# Initialize the tokenizer with a token for OOV words
tokenizer = Tokenizer(oov_token="<oov>")
tokenizer.fit_on_texts(df['comment'])
word_index = tokenizer.word_index
print(f"Size of vocabulary: {len(word_index)}")

Size of vocabulary: 7502


In [ ]:
crf_model = joblib.load("machi_v2.pkl")

In [ ]:
def tokenize_with_offsets(text):
    # Define a regex pattern to match words and non-whitespace characters
    pattern = r"\w+|[^\w\s]"
    # Use regex to find all matches in the text and return them with their start and end positions
    return [(m.group(), m.start(), m.end()) for m in re.finditer(pattern, text)]

def word2features(tokens, i):
    # Get the current word from the tokens list
    w = tokens[i]
    # Create a feature dictionary for the current word
    feats = {
        "bias": 1.0,  # Bias feature for the model
        "word.lower": w.lower(),  # Lowercase version of the word
        "word[-3:]": w[-3:],  # Last three characters of the word
        "word.isupper": w.isupper(),  # Check if the word is uppercase
        "word.istitle": w.istitle(),  # Check if the word is title case
        "word.isdigit": w.isdigit(),  # Check if the word is a digit
    }
    # If not the first word, add features for the previous word
    if i > 0:
        p = tokens[i - 1]
        feats.update({
            "-1:word.lower": p.lower(),  # Lowercase of the previous word
            "-1:word.istitle": p.istitle(),  # Title case check for the previous word
            "-1:word.isupper": p.isupper(),  # Uppercase check for the previous word
        })
    else:
        feats["BOS"] = True  # Beginning of sentence feature
    # If not the last word, add features for the next word
    if i < len(tokens) - 1:
        n = tokens[i + 1]
        feats.update({
            "+1:word.lower": n.lower(),  # Lowercase of the next word
            "+1:word.istitle": n.istitle(),  # Title case check for the next word
            "+1:word.isupper": n.isupper(),  # Uppercase check for the next word
        })
    else:
        feats["EOS"] = True  # End of sentence feature
    return feats  # Return the feature dictionary

def sent2features(tokens):
    # Generate features for each token in the sentence
    return [word2features(tokens, i) for i in range(len(tokens))]

In [ ]:
# Australia Post street types (code + full names)
STREET_TYPES = [
    "ALLY","ALLEY","ALWY","ALLEYWAY","AMBL","AMBLE","ANCG","ANCHORAGE","APP","APPROACH",
    "ARC","ARCADE","ART","ARTERY","AVE","AVENUE","BASN","BASIN","BCH","BEACH","BEND","BEND",
    "BLK","BLOCK","BVD","BOULEVARD","BRCE","BRACE","BRAE","BRAE","BRK","BREAK","BDGE","BRIDGE",
    "BDWY","BROADWAY","BROW","BROW","BYPA","BYPASS","BYWY","BYWAY","CAUS","CAUSEWAY","CTR","CENTRE",
    "CNWY","CENTREWAY","CH","CHASE","CIR","CIRCLE","CLT","CIRCLET","CCT","CIRCUIT","CRCS","CIRCUS",
    "CL","CLOSE","CLDE","COLONNADE","CMMN","COMMON","CON","CONCOURSE","CPS","COPSE","CNR","CORNER",
    "CSO","CORSO","CT","COURT","CTYD","COURTYARD","COVE","COVE","CRES","CRESCENT","CRST","CREST",
    "CRSS","CROSS","CRSG","CROSSING","CRD","CROSSROAD","COWY","CROSSWAY","CUWY","CRUISEWAY",
    "CDS","CUL-DE-SAC","CTTG","CUTTING","DALE","DALE","DELL","DELL","DEVN","DEVIATION","DIP","DIP",
    "DSTR","DISTRIBUTOR","DR","DRIVE","DRWY","DRIVEWAY","EDGE","EDGE","ELB","ELBOW","END","END",
    "ENT","ENTRANCE","ESP","ESPLANADE","EST","ESTATE","EXP","EXPRESSWAY","EXTN","EXTENSION",
    "FAWY","FAIRWAY","FTRK","FIRE","FITR","FIRETRAIL","FLAT","FLAT","FOLW","FOLLOW","FTWY","FOOTWAY",
    "FSHR","FORESHORE","FORM","FORMATION","FWY","FREEWAY","FRNT","FRONT","FRTG","FRONTAGE","GAP","GAP",
    "GDN","GARDEN","GTE","GATE","GDNS","GARDENS","GTES","GATES","GLD","GLADE","GLEN","GLEN","GRA","GRANGE",
    "GRN","GREEN","GRND","GROUND","GR","GROVE","GLY","GULLY","HTS","HEIGHTS","HRD","HIGHROAD","HWY","HIGHWAY",
    "HILL","HILL","INTG","INTERCHANGE","INTN","INTERSECTION","JNC","JUNCTION","KEY","KEY","LDG","LANDING",
    "LANE","LANE","LNWY","LANEWAY","LEES","LEES","LINE","LINE","LINK","LINK","LT","LITTLE","LKT","LOOKOUT",
    "LOOP","LOOP","LWR","LOWER","MALL","MALL","MNDR","MEANDER","MEW","MEW","MEWS","MEWS","MWY","MOTORWAY",
    "MT","MOUNT","NOOK","NOOK","OTLK","OUTLOOK","PDE","PARADE","PARK","PARK","PKLD","PARKLANDS","PKWY","PARKWAY",
    "PART","PART","PASS","PASS","PATH","PATH","PHWY","PATHWAY","PIAZ","PIAZZA","PL","PLACE","PLAT","PLATEAU",
    "PLZA","PLAZA","PKT","POCKET","PNT","POINT","PORT","PORT","PROM","PROMENADE","QUAD","QUAD","QDGL","QUADRANGLE",
    "QDRT","QUADRANT","QY","QUAY","QYS","QUAYS","RMBL","RAMBLE","RAMP","RAMP","RNGE","RANGE","RCH","REACH",
    "RES","RESERVE","REST","REST","RTT","RETREAT","RIDE","RIDE","RDGE","RIDGE","RGWY","RIDGEWAY","ROWY","RIGHT",
    "RING","RING","RISE","RISE","RVR","RIVER","RVWY","RIVERWAY","RVRA","RIVIERA","RD","ROAD","RDS","ROADS",
    "RDSD","ROADSIDE","RDWY","ROADWAY","RNDE","RONDE","RSBL","ROSEBOWL","RTY","ROTARY","RND","ROUND","RTE","ROUTE",
    "ROW","ROW","RUE","RUE","RUN","RUN","SWY","SERVICE","SDNG","SIDING","SLPE","SLOPE","SND","SOUND","SPUR","SPUR",
    "SQ","SQUARE","STRS","STAIRS","SHWY","STATE","STPS","STEPS","STRA","STRAND","ST","STREET","STRP","STRIP",
    "SBWY","SUBWAY","TARN","TARN","TCE","TERRACE","THOR","THOROUGHFARE","TLWY","TOLLWAY","TOP","TOP","TOR","TOR",
    "TWRS","TOWERS","TRK","TRACK","TRL","TRAIL","TRLR","TRAILER","TRI","TRIANGLE","TKWY","TRUNKWAY","TURN","TURN",
    "UPAS","UNDERPASS","UPR","UPPER","VALE","VALE","VDCT","VIADUCT","VIEW","VIEW","VLLS","VILLAS","VSTA","VISTA",
    "WADE","WADE","WALK","WALK","WKWY","WALKWAY","WAY","WAY","WHRF","WHARF","WYND","WYND","YARD","YARD"
]

STREET_PATTERN = re.compile(
    r"\b\d+\s+[A-Za-z0-9\s]+\s+(?:" + "|".join(STREET_TYPES) + r")\b",
    flags=re.I
)

In [ ]:
def redact_crf(text: str) -> str:
    """
    1) Early regex‐based redaction on the raw text
    2) CRF‐based redaction on the remainder
    """
    # Remove unreadable characters
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    # ── 1) Early regex fallbacks on raw text ────────────────────────────
    # email addresses (remain contiguous so this catches them)
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "[redacted]", text
    )
    # mobile numbers: 4-3-3 format
    text = re.sub(r"\b\d{4}\s\d{3}\s\d{3}\b", "[redacted]", text)
    # street addresses: "12 Elm Avenue"
    text = STREET_PATTERN.sub("[redacted]", text)
    # city/state/postcode: "Melbourne VIC 3004"
    text = re.sub(
        r"\b[A-Za-z ]+\s+(VIC|NSW|QLD|SA|WA|TAS|NT|ACT)\s*\d{4}\b",
        "[redacted]", text, flags=re.I
    )
    # 7-digit numbers
    text = re.sub(r"\b\d{7}\b", "[redacted]", text)
    # final numeric catch-alls for medicare/concession/license
    for pat in [
        r"\b\d{4}\s\d{5}\s\d\b",
        r"\b\d{3}\s\d{3}\s\d{3}[A-Za-z]\b",
        r"\b\d{9}\b"
    ]:
        text = re.sub(pat, "[redacted]", text)

    # ── 2) CRF-based redaction on what's left ────────────────────────────
    toks   = tokenize_with_offsets(text)
    words  = [w for w, _, _ in toks]
    feats  = sent2features(words)
    tags   = crf_model.predict_single(feats)

    out, i = [], 0
    while i < len(words):
        if tags[i].startswith("B-"):
            # skip the whole entity
            while i < len(words) and tags[i].startswith(("B-","I-")):
                i += 1
            out.append("[redacted]")
        else:
            out.append(words[i])
            i += 1

    return " ".join(out)

In [ ]:
# ---------------------------------------------
# Preprocessing Setup (HIMARI)
# ---------------------------------------------
def himari_preprocess(text: str, case_origin: str, stop_words: Set[str]) -> str:
    """
    1) Extract body by case_origin  
    2) lowercase + strip  
    3) redact PII via CRF (redact_crf)  
    4) remove stopwords  
    5) wrap with <start> / <end>
    """
    # --- Case‐origin extraction ---
    if case_origin == "Email":
        # Remove classic headers (From/Sent/To/Subject lines)
        text = re.sub(r'^(?:From|Sent|To|Subject):.*\r?\n', '', text, flags=re.I|re.M)
        # If an inline reply appears, cut there
        m = re.search(r'(^On .+ wrote:)', text, flags=re.I|re.M)
        if m:
            text = text[:m.start()]
        # Drop forwarded‐message markers
        text = re.split(r'-----Original Message-----', text, flags=re.I)[0]

    elif case_origin == "Web":
        m = re.search(r"Case Description:(.*?)(?=First Name:)", text, flags=re.S|re.I)
        if m:
            text = m.group(1)

    # --- Normalize ---
    text = text.lower().strip()

    # --- PII Redaction ---
    text = redact_crf(text)

    # --- Stopword removal ---
    toks     = text.split()
    filtered = [t for t in toks if t not in stop_words]

    # --- Wrap and return ---
    return "<start> " + " ".join(filtered) + " <end>"

In [ ]:
# Set model path
model_path = 'hikari_v2.keras'   # HIKARI model

# Load HIKARI model
if os.path.exists(model_path):
    if os.path.isfile(model_path):
        try:
            hikari_model = load_model(model_path, compile=False)
            print("HIKARI model loaded successfully.")
            print("Model type:", type(hikari_model))  # Should be Sequential or Functional
            print("Input shape:", hikari_model.input_shape)
        except Exception as e:
            print(f"Error loading HIKARI model: {e}")
    else:
        print(f"{model_path} is a directory, not a file. Expected a .keras single file.")
else:
    print(f"Model file not found at: {model_path}")

HIKARI model loaded successfully.
Model type: <class 'keras.src.engine.sequential.Sequential'>
Input shape: (None, 196)


In [ ]:
# ---------------------------------------------
#  Prediction Function
# ---------------------------------------------

def predict_drivers_from_comment(
    raw_comment: str,
    case_origin: str = "Other",
    threshold: float = 0.5
):
    """
    Preprocess and predict drivers from a raw customer comment.
    """
    # Preprocess (now passing both origin & stop_words)
    cleaned_comment = himari_preprocess(raw_comment, case_origin, stop_words)

    # Convert to sequence
    sequence = tokenizer.texts_to_sequences([cleaned_comment])
    padded   = pad_sequences(sequence, maxlen=196, padding='post', truncating='post')

    # Predict
    probabilities = hikari_model.predict(padded, verbose=0)[0]

    # Interpret results
    predicted_drivers = []
    topics = [
        'Customer Service','Digital','Online Experience','Outages and Faults',
        'Process','Reputation','Sustainability','Trust',
        'Value for Money','Vulnerability & FDV'
    ]
    for idx, prob in enumerate(probabilities):
        if prob >= threshold:
            predicted_drivers.append((topics[idx], round(float(prob), 3)))

    # Sort by probability
    predicted_drivers.sort(key=lambda x: x[1], reverse=True)

    return predicted_drivers, cleaned_comment

In [ ]:
print("\nHIMARI + HIKARI UAT Mode")
print("Type a customer comment to see preprocessing + driver detection.")
print("Type 'exit' to quit.\n")

while True:
    case_origin = input("Enter case origin (Email/Web/Other): ").strip()
    if case_origin.lower() == 'exit':
        print("Goodbye!")
        break

    raw_comment = input("Enter customer comment: ").strip()
    if raw_comment.lower() == 'exit':
        print("Goodbye!")
        break

    drivers, cleaned = predict_drivers_from_comment(raw_comment, case_origin)

    print(f"\nPreprocessed Comment: {cleaned}")
    if drivers:
        print("Predicted Drivers:")
        for driver, score in drivers:
            print(f" - {driver} (Confidence: {score})")
    else:
        print("No drivers detected confidently.")
    print("\n---\n")


HIMARI + HIKARI UAT Mode
Type a customer comment to see preprocessing + driver detection.
Type 'exit' to quit.



Enter case origin (Email/Web/Other):  Email

Enter customer comment:  From: Sarah Lin <sarah.lin@gmail.com> Sent: Tuesday, May 1, 2025 10:05 AM To: helpdesk@powerco.com Subject: Problem with Online Account  Hello PowerCo Team,  I’m having trouble accessing my online account dashboard. Every time I log in, I get an “Invalid credentials” error. My billing number is 1234567 and my phone number is 0412 345 678. I live at 12 Elm Avenue, Melbourne VIC 3004, and my account number is 567890123. Could you please reset my password or let me know what’s wrong?  Thank you,   Sarah Lin  -----Original Message-----   From: PowerCo Support <support@powerco.com>   Sent: Monday, April 30, 2025 09:30 PM   To: Sarah Lin <sarah.lin@gmail.com>   Subject: Re: Problem with Online Account  


Preprocessed Comment: <start> : sarah lin < [ redacted ] > sent : tuesday , may 1 , 2025 10 : 05 : [ redacted ] subject : problem online account hello powerco team , im trouble accessing online account dashboard . every time log , get invalid credentials error . billing number [redacted] phone number [redacted] . live [redacted] , [redacted] , account number [redacted] . could please reset password let know whats wrong ? thank , sarah lin <end>
Predicted Drivers:
 - Online Experience (Confidence: 0.998)
 - Value for Money (Confidence: 0.995)
 - Customer Service (Confidence: 0.91)
 - Digital (Confidence: 0.909)

---



Enter case origin (Email/Web/Other):  Web

...

Enter case origin (Email/Web/Other):  Web

Enter customer comment:  Case Description: look here u lot your crew was hammering the pipes at 5AM this morn, WOKE ME UP MID‐DREAM and ive got a big job today gotta be on time. couldnt get back to sleep, tools left everywhere and water dripped on my laptop. fix your schedule or imma report u.  First name: Jake   Last name: Turner   Phone: 0455 123 987   Email: j.turner@buildermail.com   Address: 88 Industrial Road, Sunshine VIC 3020


Preprocessed Comment: <start> look u lot crew hammering pipes 5am morn , woke middream ive got big job today gotta time . couldnt get back sleep , tools left everywhere dripped laptop . fix schedule imma report u . <end>
Predicted Drivers:
 - Value for Money (Confidence: 1.0)

---



Enter case origin (Email/Web/Other):  exit

Goodbye!


From: Sarah Lin <sarah.lin@gmail.com>
Sent: Tuesday, May 1, 2025 10:05 AM
To: helpdesk@powerco.com
Subject: Problem with Online Account

Hello PowerCo Team,

I’m having trouble accessing my online account dashboard. Every time I log in, I get an “Invalid credentials” error. My billing number is 1234567 and my phone number is 0412 345 678. I live at 12 Elm Avenue, Melbourne VIC 3004, and my account number is 567890123. Could you please reset my password or let me know what’s wrong?

Thank you,  
Sarah Lin

-----Original Message-----  
From: PowerCo Support <support@powerco.com>  
Sent: Monday, April 30, 2025 09:30 PM  
To: Sarah Lin <sarah.lin@gmail.com>  
Subject: Re: Problem with Online Account  

> Hi Sarah,  
> Could you please confirm your best contact number and postal address?  
> Thanks,  
> The PowerCo Team

Case Description:
I’m absolutely appalled at this month’s bill – I checked it three times and it’s gone up by over $200 even though I barely used the hot water! At my age, every dollar counts and I don’t understand how you can charge me so much for what’s supposed to be a “concession rate.” I’ve been with this company for over 40 years and never had a surprise like this. Please explain immediately or reverse these charges. I am on a fixed pension and can’t afford this.

First name: Margaret  
Last name: O’Reilly  
Date of birth: 15/07/1945  
Address: 22 Elder Street, Glenroy VIC 3046  
Phone: 0412 111 222  
Email: margaret.oreilly@example.com  
Account number: 7654321

From: Alex Turner <alex.turner@startupmail.com>  
Sent: Wednesday, May 7, 2025 2:14 PM  
To: support@waterco.com  
Subject: Unsatisfactory Contractor Visit

Hello Support Team,

I’m writing because yesterday’s contractor visit was completely unsatisfactory. I booked a time slot between 9–11 AM and they didn’t arrive until almost 1 PM—and then they spent only 10 minutes fixing a dripping tap before rushing off. The leak has already started again this morning. As a busy professional, I took time off work for this and felt the service level was unacceptable.

Please arrange for a proper follow-up call and a full re-inspection at a time that actually works, or advise how to lodge a formal complaint.

Thank you,  
Alex Turner  
Phone: 0488 999 777  
Address: 45 Innovation Drive, Docklands VIC 3008  
Customer ID: 99887760

Caller Ms Jane Smith rang at 10:22 AM to report a water leak under the kitchen sink. She says the water has been dripping steadily overnight, causing damp patches on the cupboard floor. Reported noticing a strange smell this morning and is worried about mould. Advised her that a technician will be out within 24 hours. Escalated to urgent leak Crew.

Name: Jane Smith  
Phone: 03 9800 4567  
Address: 12 Riverside Road, Footscray VIC 3011  
Account #: 99887766

From: EILEEN JOHNSON <e.johnson1948@oldmail.com>  
Sent: Monday, May 12, 2025 08:47 AM  
To: support@utilityco.com  
Subject: POSTBOX BROKED BY UR CONTRACTOR

HI THERE, I DONT KNOW IF IM DOING THIS RITE BUT YOUR GUY CAME YESTERDAY AND HE BROKED MY POSTBOX OFF THE WALL. I DIDNT HEAR ANYTHING TILL I WENT TO GET MY MAIL THIS MORNING AND IT WAS LYING ON THE GROUND. I TRIED TO FIX IT BUT MY HANDS DONT WORK LIKE THEY USED TO. I NEED IT SORTED BEFORE I MISS IMPORTANT LETTERS.

PLS SEND SOMEONE WHO KNOWS WHAT THEIR DOIN. I DONT WANT NO MORE BILLS FOR THIS.

THANKS,  
EILEEN JOHNSON  
PHONE: 0417 888 999  
ADDRESS: 5 OLD SCHOOL LANE, MALVERN VIC 3144  
ACCOUNT: 99887766

Case Description:
look here u lot your crew was hammering the pipes at 5AM this morn, WOKE ME UP MID‐DREAM and ive got a big job today gotta be on time. couldnt get back to sleep, tools left everywhere and water dripped on my laptop. fix your schedule or imma report u.

First name: Jake  
Last name: Turner  
Phone: 0455 123 987  
Email: j.turner@buildermail.com  
Address: 88 Industrial Road, Sunshine VIC 3020

Customer angry said contractors woke her at 2am banging pipes banging banging. She’s screaming saying can’t live like this. Wants refund, threatens to cancel service if no apology.

Name: Patricia Lee  
Phone: 0412 654 321  
Address: 14 Nightingale Street, Ivanhoe VIC 3079  
Account#: 55667788